# 01 - Data Understanding

**Customer Churn Prediction in the Telecommunications Industry**
AAI-510, Group 6, University of San Diego

## Problem statement and justification

Telecom companies lose revenue when customers leave (churn). Retaining an existing
customer is far cheaper than acquiring a new one, so the business needs an early-warning
system that flags customers who are likely to churn. That lets retention teams act before
the customer leaves, for example with a targeted offer or a contract upgrade.

We frame this as a **binary classification** problem: predict whether a customer will
churn (Yes) or stay (No), using demographic, contract, payment, and billing features.
Because the cost of missing a real churner is higher than the cost of a false alarm, we
will pay special attention to **Recall** and **ROC-AUC** later in the project.

This first notebook focuses on understanding the data before any modeling: what columns we
have, their types and ranges, whether the data is clean, and how the target is balanced.


## Load the data and inspect its structure

We load the synthetic Telco churn dataset and look at its shape, column types, and a sample
of rows. The file lives one directory up from the notebooks folder.


In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

# Load the raw synthetic dataset (one level up from notebooks/)
DATA_PATH = "../data/synthetic_customer_churn_100k.csv"
df = pd.read_csv(DATA_PATH)

print(f"Rows, columns: {df.shape}")
df.head()

Rows, columns: (100000, 9)


,CustomerID,Age,Gender,Tenure,MonthlyCharges,Contract,PaymentMethod,TotalCharges,Churn
0,1,56,Female,68,147.58,Two year,Bank transfer,10052.03,No
1,2,69,Male,32,22.54,Month-to-month,Mailed check,686.78,No
2,3,46,Female,10,52.47,One year,Electronic check,537.88,No
3,4,32,Male,22,109.67,Month-to-month,Mailed check,2390.04,Yes
4,5,60,Female,54,130.98,Month-to-month,Credit card,7081.28,No


In [2]:
# Column names, non-null counts, and dtypes
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100000 entries, 0 to 99999
Data columns (total 9 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   CustomerID      100000 non-null  int64  
 1   Age             100000 non-null  int64  
 2   Gender          100000 non-null  object 
 3   Tenure          100000 non-null  int64  
 4   MonthlyCharges  100000 non-null  float64
 5   Contract        100000 non-null  object 
 6   PaymentMethod   100000 non-null  object 
 7   TotalCharges    100000 non-null  float64
 8   Churn           100000 non-null  object 
dtypes: float64(2), int64(3), object(4)
memory usage: 6.9+ MB


### Data dictionary

The dataset is fully synthetic (no real customer information) and is documented on Kaggle.
The nine columns are:

| Column | Description | Type |
| --- | --- | --- |
| CustomerID | Unique customer identifier, dropped before modeling | int |
| Age | Customer age in years (18 to 80) | int |
| Gender | Female, Male, or Other | string |
| Tenure | Months with the company (1 to 72) | int |
| MonthlyCharges | Monthly bill in USD (about 10 to 150) | float |
| TotalCharges | Total billed over tenure | float |
| Contract | Month-to-month, One year, or Two year | string |
| PaymentMethod | Bank transfer, Credit card, Electronic check, or Mailed check | string |
| Churn | Target: Yes if the customer left, No otherwise | string |


## Data quality checks

Before trusting the data we check for missing values and duplicate rows. We also confirm
the customer identifier is unique, since it should be a primary key.


In [3]:
# Missing values per column
missing = df.isnull().sum()
print("Missing values per column:")
print(missing)
print(f"\nTotal missing values: {int(missing.sum())}")

Missing values per column:
CustomerID        0
Age               0
Gender            0
Tenure            0
MonthlyCharges    0
Contract          0
PaymentMethod     0
TotalCharges      0
Churn             0
dtype: int64

Total missing values: 0


In [4]:
# Duplicate rows and CustomerID uniqueness
print(f"Fully duplicated rows: {int(df.duplicated().sum())}")
print(f"Duplicated CustomerID values: {int(df['CustomerID'].duplicated().sum())}")
print(f"Unique customers: {df['CustomerID'].nunique():,} of {len(df):,} rows")

Fully duplicated rows: 0
Duplicated CustomerID values: 0
Unique customers: 100,000 of 100,000 rows


## Summary statistics

We describe the numeric columns to confirm the ranges match the documented data dictionary,
then look at the distribution of each categorical column.


In [5]:
# Numeric summary
num_cols = ["Age", "Tenure", "MonthlyCharges", "TotalCharges"]
df[num_cols].describe().round(2)

,Age,Tenure,MonthlyCharges,TotalCharges
count,100000.00,100000.00,100000.00,100000.00
mean,49.03,36.53,79.97,2926.14
std,18.18,20.79,40.49,2388.16
min,18.00,1.00,10.00,-118.43
25%,33.00,18.00,44.72,963.67
50%,49.00,37.00,80.00,2268.06
75%,65.00,54.00,115.05,4394.33
max,80.00,72.00,150.00,10831.46


In [6]:
# Categorical value counts
cat_cols = ["Gender", "Contract", "PaymentMethod"]
for col in cat_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

--- Gender ---
Gender
Female    48256
Male      47787
Other      3957
Name: count, dtype: int64

--- Contract ---
Contract
Month-to-month    54915
One year          25261
Two year          19824
Name: count, dtype: int64

--- PaymentMethod ---
PaymentMethod
Electronic check    34892
Mailed check        25221
Credit card         20032
Bank transfer       19855
Name: count, dtype: int64



## Target balance

The target is `Churn`. We check how many customers churned versus stayed. The Kaggle page
describes the churn rate as about 20 percent, so we verify the actual rate in this file.


In [7]:
churn_counts = df["Churn"].value_counts()
churn_rate = df["Churn"].eq("Yes").mean()
print(churn_counts)
print(f"\nMeasured churn rate: {churn_rate:.1%}")

Churn
No     66856
Yes    33144
Name: count, dtype: int64

Measured churn rate: 33.1%


### Inference and takeaways

- The dataset has **100,000 rows and 9 columns**, with **no missing values** and **no
  duplicate rows**. `CustomerID` is unique, so the data is clean and ready for analysis.
- Numeric ranges match the documentation: Age 18 to 80, Tenure 1 to 72 months,
  MonthlyCharges roughly 10 to 150, and TotalCharges scaling with tenure and monthly charge.
- `Gender` has three categories (Female, Male, Other), where Other is a small share of
  customers. `Contract` and `PaymentMethod` are clean categoricals with no unexpected values.
- The **measured churn rate is about 33 percent**, higher than the roughly 20 percent the
  Kaggle page describes. We treat this as a mildly imbalanced problem and report the
  measured rate throughout. The imbalance is modest, so resampling is optional.
- Because the data is clean, the next notebook can move directly to exploratory analysis of
  how each feature relates to churn.
